# 02 — Transformer untuk Teks: Membangun Mini-GPT dari Nol

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riki-profile/ai-engineer-journey/blob/main/06-transformers-nlp/02_transformer_decoder_dari_nol.ipynb)

Di modul 05 kita membangun **encoder** Transformer untuk gambar. Sekarang kita membangun **decoder**: model yang menebak **token berikutnya**, prinsip yang sama dengan GPT dan semua LLM modern (modul 08).

Isi notebook:
1. Tiga keluarga Transformer: **encoder-only** (BERT), **decoder-only** (GPT), dan **encoder–decoder** (T5).
2. **Causal mask**: mencegah model "mencontek" token masa depan.
3. **Positional encoding** sinusoidal.
4. Melatih **mini-GPT tingkat karakter** pada teks Shakespeare (≈1 MB, domain publik).
5. **Generasi teks** dengan *temperature* dan *top-k sampling*.

Sebagian kode attention diambil dari modul 05, jadi pastikan konsep Q, K, V di sana sudah dipahami.

## 0. Cek GPU

Di Colab pilih **Runtime → Change runtime type → T4 GPU**, lalu jalankan ulang dari atas. Notebook tetap bisa berjalan di CPU, tetapi jauh lebih lambat.

In [ ]:
import shutil
import subprocess
import torch

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"✅ GPU terdeteksi: {props.name} | memori {props.total_memory / 1e9:.1f} GB")
else:
    print("⚠️ GPU tidak terdeteksi, akan berjalan di CPU (lebih lambat).")
    print("   Di Colab: Runtime → Change runtime type → T4 GPU")

if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"])

## 1. Tiga Keluarga Transformer

| | **Encoder-only** | **Decoder-only** | **Encoder–decoder** |
|---|---|---|---|
| Contoh | BERT, RoBERTa, IndoBERT | GPT, LLaMA, Claude | T5, BART, mT5 |
| Attention | dua arah (setiap token melihat semua token) | **kausal** (hanya melihat token sebelumnya) | encoder dua arah + decoder kausal + *cross-attention* ke encoder |
| Tujuan pretraining | *Masked LM*: tebak token yang ditutup `[MASK]` | *Causal LM*: tebak token berikutnya | *Seq2seq*: ubah teks input menjadi teks output |
| Cocok untuk | klasifikasi, NER, embedding (modul 07) | generasi teks, chat, LLM (modul 08–11) | terjemahan, ringkasan |

Perbedaan utama mereka hanya pada **mask attention**. Mari lihat bentuk mask-nya.

In [ ]:
import math
import os
import time
import urllib.request
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

T = 6
mask_dua_arah = torch.ones(T, T)
mask_kausal = torch.tril(torch.ones(T, T))        # segitiga bawah: token i hanya melihat 0..i

fig, ax = plt.subplots(1, 2, figsize=(9, 4))
for a, m, judul in [(ax[0], mask_dua_arah, "Encoder (BERT): dua arah"), (ax[1], mask_kausal, "Decoder (GPT): kausal")]:
    a.imshow(m, cmap="Greens", vmin=0, vmax=1)
    a.set_title(judul); a.set_xlabel("token yang dilihat (key)"); a.set_ylabel("token yang memproses (query)")
    a.set_xticks(range(T)); a.set_yticks(range(T))
plt.tight_layout(); plt.show()

## 2. Causal Self-Attention

Saat training, kita memberi model seluruh kalimat sekaligus dan memintanya menebak token berikutnya **di setiap posisi** secara paralel. Tanpa mask, posisi ke-3 bisa "melihat" jawabannya di posisi ke-4.

Caranya: sebelum softmax, skor untuk posisi masa depan diisi **−∞**, sehingga setelah softmax bobotnya menjadi **0**.

Kita cek implementasi manual terhadap `F.scaled_dot_product_attention(..., is_causal=True)`, versi PyTorch yang jauh lebih cepat (memakai *FlashAttention* di GPU yang mendukung).

In [ ]:
q = torch.randn(1, T, 8); k = torch.randn(1, T, 8); v = torch.randn(1, T, 8)

skor = q @ k.transpose(-2, -1) / math.sqrt(8)
skor = skor.masked_fill(mask_kausal == 0, float("-inf"))   # tutup masa depan
bobot = torch.softmax(skor, dim=-1)
manual = bobot @ v

torch.set_printoptions(precision=2, sci_mode=False)
print("bobot attention (segitiga bawah, tiap baris berjumlah 1):\n", bobot[0])
print("\nsama dengan F.scaled_dot_product_attention(is_causal=True)?",
      torch.allclose(manual, F.scaled_dot_product_attention(q, k, v, is_causal=True), atol=1e-6))

## 3. Positional Encoding Sinusoidal

Di ViT (modul 05) posisi berupa vektor yang **dipelajari**. Paper Transformer asli (2017) memakai rumus tetap: setiap dimensi adalah gelombang sinus/cosinus dengan frekuensi berbeda.

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d}}\right) \qquad PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d}}\right)$$

Kelebihannya: tidak ada parameter, dan bisa dihitung untuk posisi berapa pun. LLM modern umumnya memakai varian lain (misalnya **RoPE**, yang memutar Q dan K), tetapi idenya sama: menyuntikkan informasi urutan.

In [ ]:
def sinusoidal_pe(n_pos, d):
    pos = torch.arange(n_pos).unsqueeze(1)
    frek = torch.exp(torch.arange(0, d, 2) * (-math.log(10000.0) / d))
    pe = torch.zeros(n_pos, d)
    pe[:, 0::2] = torch.sin(pos * frek)
    pe[:, 1::2] = torch.cos(pos * frek)
    return pe

pe = sinusoidal_pe(128, 64)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].imshow(pe.T, aspect="auto", cmap="RdBu"); ax[0].set_xlabel("posisi"); ax[0].set_ylabel("dimensi")
ax[0].set_title("Positional encoding (128 posisi × 64 dimensi)")
for d in [0, 10, 30]:
    ax[1].plot(pe[:, d], label=f"dimensi {d}")
ax[1].set_xlabel("posisi"); ax[1].legend(); ax[1].set_title("Dimensi awal = frekuensi tinggi, akhir = rendah")
plt.tight_layout(); plt.show()

## 4. Data: Teks Shakespeare

Kita pakai *tiny Shakespeare* (≈1 juta karakter, domain publik), dataset klasik untuk model bahasa kecil. Tokenisasinya **tingkat karakter** (vocab ≈ 65) supaya fokus ke arsitektur; di notebook 01 kita sudah melihat bahwa model sungguhan memakai subword.

`QUICK_RUN=1` memakai model dan jumlah langkah yang sangat kecil untuk uji cepat.

In [ ]:
QUICK_RUN = os.environ.get("QUICK_RUN") == "1"
URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
if not os.path.exists("shakespeare.txt"):
    urllib.request.urlretrieve(URL, "shakespeare.txt")
teks = open("shakespeare.txt", encoding="utf-8").read()

chars = sorted(set(teks))
VOCAB = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]          # noqa: E731
decode = lambda ids: "".join(itos[i] for i in ids)  # noqa: E731

data = torch.tensor(encode(teks), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]
print(f"{len(teks):,} karakter | vocab {VOCAB} | contoh:\n{teks[:200]}")

### 4.1 Membuat batch

Setiap contoh training adalah potongan acak sepanjang `BLOCK` karakter. **Target** = input yang digeser satu posisi ke kanan: di setiap posisi, model menebak karakter berikutnya. Satu potongan sepanjang `BLOCK` menghasilkan `BLOCK` contoh training sekaligus, berkat causal mask.

In [ ]:
BLOCK = 32 if QUICK_RUN else 128     # panjang konteks
BATCH = 16 if QUICK_RUN else 64

def get_batch(split):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - BLOCK - 1, (BATCH,))
    x = torch.stack([d[i:i + BLOCK] for i in ix])
    y = torch.stack([d[i + 1:i + BLOCK + 1] for i in ix])
    return x.to(device), y.to(device)

x, y = get_batch("train")
print("input :", repr(decode(x[0, :20].tolist())))
print("target:", repr(decode(y[0, :20].tolist())), "← digeser 1 karakter")

## 5. Arsitektur Mini-GPT

Strukturnya hampir sama dengan blok encoder di modul 05. Bedanya: attention memakai **causal mask**, dan output-nya adalah **logits untuk setiap token di vocab** di setiap posisi (bukan satu kelas untuk seluruh input).

```
token ids → token embedding + position embedding
          → N × [x + CausalAttention(LN(x)) ; x + MLP(LN(x))]
          → LayerNorm → Linear(dim → VOCAB) → logits per posisi
```

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, dim, n_heads, dropout):
        super().__init__()
        self.n_heads = n_heads
        self.qkv = nn.Linear(dim, 3 * dim)
        self.proj = nn.Linear(dim, dim)
        self.dropout = dropout

    def forward(self, x):
        B, T, D = x.shape
        q, k, v = self.qkv(x).split(D, dim=2)
        q, k, v = (t.view(B, T, self.n_heads, D // self.n_heads).transpose(1, 2) for t in (q, k, v))
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True,
                                             dropout_p=self.dropout if self.training else 0.0)
        return self.proj(out.transpose(1, 2).reshape(B, T, D))


class Block(nn.Module):
    def __init__(self, dim, n_heads, dropout):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.attn = CausalSelfAttention(dim, n_heads, dropout)
        self.mlp = nn.Sequential(nn.Linear(dim, 4 * dim), nn.GELU(), nn.Linear(4 * dim, dim), nn.Dropout(dropout))

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        return x + self.mlp(self.ln2(x))


class MiniGPT(nn.Module):
    def __init__(self, vocab, block, dim=192, depth=4, n_heads=6, dropout=0.1):
        super().__init__()
        self.block = block
        self.tok_emb = nn.Embedding(vocab, dim)
        self.pos_emb = nn.Embedding(block, dim)          # posisi dipelajari (seperti GPT-2)
        self.blocks = nn.Sequential(*[Block(dim, n_heads, dropout) for _ in range(depth)])
        self.ln_f = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, vocab)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.tok_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device))
        logits = self.head(self.ln_f(self.blocks(x)))            # (B, T, VOCAB)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss


cfg = dict(dim=64, depth=2, n_heads=4) if QUICK_RUN else dict(dim=192, depth=4, n_heads=6)
model = MiniGPT(VOCAB, BLOCK, **cfg).to(device)
print(f"Parameter: {sum(p.numel() for p in model.parameters()):,}")

_, loss = model(*get_batch("train"))
print(f"Loss awal: {loss.item():.3f} | tebakan acak: ln({VOCAB}) = {math.log(VOCAB):.3f}")

Loss awal mendekati `ln(VOCAB)`: model yang belum belajar memberi peluang kurang lebih sama ke semua karakter. Ini cara cepat mengecek bahwa model dan loss sudah terpasang dengan benar.

## 6. Training

Karena data berupa aliran teks tanpa "epoch" alami, training diukur dalam **langkah** (*iterations*). Setiap `EVAL_EVERY` langkah, loss dirata-ratakan pada beberapa batch train dan validasi.

In [ ]:
STEPS = 60 if QUICK_RUN else 3000
EVAL_EVERY = 20 if QUICK_RUN else 250
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.1)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=STEPS)


@torch.no_grad()
def estimasi_loss(n_batch=10):
    model.eval()
    hasil = {s: torch.stack([model(*get_batch(s))[1] for _ in range(n_batch)]).mean().item() for s in ["train", "val"]}
    model.train()
    return hasil


riwayat = []
mulai = time.time()
for step in range(STEPS + 1):
    if step % EVAL_EVERY == 0:
        est = estimasi_loss()
        riwayat.append((step, est["train"], est["val"]))
        print(f"langkah {step:5d} | train {est['train']:.3f} | val {est['val']:.3f} | {time.time() - mulai:.0f} detik")
    if step == STEPS:
        break
    _, loss = model(*get_batch("train"))
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)   # cegah gradien meledak
    optimizer.step(); optimizer.zero_grad(); scheduler.step()

s, tr, va = zip(*riwayat)
plt.plot(s, tr, "o-", label="train"); plt.plot(s, va, "o-", label="val")
plt.axhline(math.log(VOCAB), ls="--", c="gray", label="tebakan acak")
plt.xlabel("langkah"); plt.ylabel("cross-entropy"); plt.legend(); plt.title("Loss mini-GPT"); plt.show()

## 7. Generasi Teks

Model menghasilkan teks **satu token per langkah** (*autoregressive*): prediksi token berikutnya → sampel → tambahkan ke input → ulangi.

Dua parameter sampling yang juga ada di API LLM (modul 08):
- **Temperature**: logits dibagi `T` sebelum softmax. `T < 1` membuat distribusi lebih tajam (konservatif, berulang); `T > 1` membuatnya lebih datar (kreatif, tapi mudah ngawur).
- **Top-k**: hanya `k` token dengan peluang tertinggi yang boleh dipilih.

Karena konteks model terbatas (`BLOCK`), hanya `BLOCK` token terakhir yang dimasukkan ke model.

In [ ]:
@torch.no_grad()
def generate(prompt, n_token=300, temperature=1.0, top_k=None):
    model.eval()
    idx = torch.tensor([encode(prompt)], device=device)
    for _ in range(n_token):
        logits, _ = model(idx[:, -BLOCK:])
        logits = logits[:, -1, :] / temperature               # hanya posisi terakhir
        if top_k is not None:
            batas = torch.topk(logits, top_k).values[:, -1:]
            logits = logits.masked_fill(logits < batas, float("-inf"))
        berikut = torch.multinomial(torch.softmax(logits, dim=-1), num_samples=1)
        idx = torch.cat([idx, berikut], dim=1)
    return decode(idx[0].tolist())


torch.manual_seed(0)
for temp in [0.5, 1.0, 1.5]:
    print(f"===== temperature {temp} =====")
    print(generate("ROMEO:\n", n_token=250, temperature=temp, top_k=20))
    print()

Dengan 3.000 langkah di GPU, model sudah menangkap **struktur**: nama tokoh diikuti titik dua, baris pendek, dan kata-kata yang terlihat seperti bahasa Inggris kuno, meskipun maknanya belum nyambung. LLM sungguhan memakai prinsip yang persis sama, dengan subword tokenizer, miliaran parameter, dan triliunan token.

## Ringkasan

- **Encoder-only** (BERT) memakai attention dua arah → pemahaman teks; **decoder-only** (GPT) memakai **causal mask** → generasi; **encoder–decoder** (T5) → seq2seq.
- Causal mask mengisi skor masa depan dengan −∞ sebelum softmax, sehingga model bisa dilatih menebak token berikutnya di semua posisi secara paralel.
- Posisi disuntikkan lewat positional encoding (sinusoidal, dipelajari, atau RoPE).
- Target training = input yang digeser satu token; loss awal ≈ `ln(vocab)`.
- Generasi autoregressive dengan **temperature** dan **top-k** mengatur keseimbangan antara konsisten dan kreatif.

## Latihan

1. **Skala model.** Latih tiga ukuran model (misalnya `dim` 64/128/256 dengan `depth` 2/4/6) dengan jumlah langkah yang sama. Catat jumlah parameter, loss validasi akhir, dan waktu training dalam tabel, lalu bandingkan kualitas teks yang dihasilkan. Apakah model terbesar selalu terbaik untuk data 1 MB ini? (petunjuk: perhatikan jarak train–val.)
2. **Bahasa Indonesia.** Ganti korpus dengan teks berbahasa Indonesia, misalnya gabungan semua kalimat NusaX (`train.csv`, `valid.csv`, `test.csv`) atau teks domain publik lain. Latih ulang dan bandingkan hasilnya. Apa masalahnya jika data hanya ~100 KB?
3. **Top-p (nucleus) sampling.** Implementasikan *top-p sampling*: urutkan peluang, ambil token terkecil yang jumlah peluangnya ≥ `p`, lalu sampel di antaranya. Bandingkan hasil `top_k=20` dengan `top_p=0.9` pada temperature yang sama, lalu jelaskan kapan top-p lebih masuk akal daripada top-k.